In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Day 7 Feature Engineered dataset
feature_file_path = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Feature_Engineered.xlsx"
)

print("Input file:")
print(feature_file_path)
print("File exists:", feature_file_path.exists())

Input file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Feature_Engineered.xlsx
File exists: True


In [3]:
features = pd.read_excel(
    feature_file_path,
    sheet_name="Feature_Engineered"
)

print("Dataset loaded successfully.")
print("Shape:", features.shape)

Dataset loaded successfully.
Shape: (10000, 50)


## Inspecting the columns

In [4]:
print("Available columns:")
    
for i, col in enumerate(    features.columns, start=1):
    print(f"{i}. {col}")

Available columns:
1. TransactionID
2. UserID
3. CourseID
4. TransactionDate
5. Amount
6. PaymentMethod
7. TeacherID
8. UserName
9. UserAge
10. UserGender
11. Email
12. CourseName
13. CourseCategory
14. CourseType
15. CourseLevel
16. CoursePrice
17. CourseDuration
18. CourseRating
19. TeacherName
20. TeacherAge
21. TeacherGender
22. Expertise
23. YearsOfExperience
24. TeacherRating
25. TransactionYear
26. TransactionMonth
27. TransactionDay
28. TransactionDayOfWeek
29. TransactionQuarter
30. MonthSin
31. MonthCos
32. IsWeekend
33. PriceDifference
34. DiscountAmount
35. DiscountPercentage
36. HasDiscount
37. CourseDurationLog
38. HighRatedCourse
39. CoursePriceLog
40. ExperienceLevel
41. HighlyExperiencedTeacher
42. HighRatedTeacher
43. UserAgeGroup
44. AdultUser
45. PriceRatingInteraction
46. ExperienceRatingInteraction
47. PricePerDuration
48. TeacherCourseRatingInteraction
49. CoursePreviousDemand
50. TeacherPreviousTransactions


In [5]:
required_columns = [
    "CourseID",
    "Amount",
    "CourseName",
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseRating",
    "TeacherID"
]

missing_columns = [
    col for col in required_columns
    if col not in features.columns
]

print("Missing required columns:")
print(missing_columns)

Missing required columns:
[]


## Checking CourseID

In [6]:
print("Number of unique courses:", features["CourseID"].nunique())
print("\nCourseID sample:")
print(features["CourseID"].head())

Number of unique courses: 60

CourseID sample:
0    CR00050
1    CR00021
2    CR00009
3    CR00021
4    CR00022
Name: CourseID, dtype: str


## Checking transaction dates

In [7]:
features["TransactionDate"] = pd.to_datetime(
    features["TransactionDate"]
)

print("Minimum transaction date:")
print(features["TransactionDate"].min())

print("\nMaximum transaction date:")
print(features["TransactionDate"].max())

Minimum transaction date:
2025-01-01 00:00:00

Maximum transaction date:
2025-12-30 00:00:00


In [8]:
print(
    features["TransactionDate"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

TransactionDate
2025-01    833
2025-02    766
2025-03    857
2025-04    831
2025-05    825
2025-06    899
2025-07    826
2025-08    844
2025-09    870
2025-10    762
2025-11    869
2025-12    818
Freq: M, Name: count, dtype: int64


## Creating Enrollmentcount

In [9]:
enrollment_target = (
    features
    .groupby("CourseID")
    .size()
    .reset_index(name="EnrollmentCount")
)

enrollment_target.head()

,CourseID,EnrollmentCount
0,CR00001,164
1,CR00002,149
2,CR00003,173
3,CR00004,154
4,CR00005,166


## Validating Enrollmentcount

In [10]:
print("Number of courses:", enrollment_target.shape[0])

print("\nEnrollmentCount summary:")
print(enrollment_target["EnrollmentCount"].describe())

Number of courses: 60

EnrollmentCount summary:
count     60.000000
mean     166.666667
std       12.523424
min      140.000000
25%      157.500000
50%      166.000000
75%      174.750000
max      196.000000
Name: EnrollmentCount, dtype: float64


In [11]:
print(
    "Total EnrollmentCount:",
    enrollment_target["EnrollmentCount"].sum()
)

print(
    "Original transaction rows:",
    len(features)
)

Total EnrollmentCount: 10000
Original transaction rows: 10000


## Creating Courserevenue

In [12]:
revenue_target = (
    features
    .groupby("CourseID")["Amount"]
    .sum()
    .reset_index(name="CourseRevenue")
)

revenue_target.head()

,CourseID,CourseRevenue
0,CR00001,77453.92
1,CR00002,0.00
2,CR00003,0.00
3,CR00004,0.00
4,CR00005,0.00


## Validating Courserevenue

In [13]:
print("Number of courses:", revenue_target.shape[0])

print("\nCourseRevenue summary:")
print(revenue_target["CourseRevenue"].describe())

Number of courses: 60

CourseRevenue summary:
count       60.000000
mean     15188.724500
std      25409.721911
min          0.000000
25%          0.000000
50%          0.000000
75%      20858.060000
max      85416.600000
Name: CourseRevenue, dtype: float64


In [14]:
print(
    "Total CourseRevenue:",
    revenue_target["CourseRevenue"].sum()
)

print(
    "Total Amount in original dataset:",
    features["Amount"].sum()
)

Total CourseRevenue: 911323.47
Total Amount in original dataset: 911323.47


## Combining the two targets

In [15]:
targets = enrollment_target.merge(
    revenue_target,
    on="CourseID",
    how="inner"
)

targets.head()

,CourseID,EnrollmentCount,CourseRevenue
0,CR00001,164,77453.92
1,CR00002,149,0.00
2,CR00003,173,0.00
3,CR00004,154,0.00
4,CR00005,166,0.00


## Validating target datasets

In [16]:
print("Target dataset shape:")
print(targets.shape)

print("\nTarget columns:")
print(targets.columns.tolist())

print("\nMissing values:")
print(targets.isnull().sum())

print("\nDuplicate CourseIDs:")
print(targets["CourseID"].duplicated().sum())

Target dataset shape:
(60, 3)

Target columns:
['CourseID', 'EnrollmentCount', 'CourseRevenue']

Missing values:
CourseID           0
EnrollmentCount    0
CourseRevenue      0
dtype: int64

Duplicate CourseIDs:
0


## Adding useful course information

In [17]:
course_info_columns = [
    "CourseID",
    "CourseName",
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherID",
    "TeacherName",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

course_info = (
    features[course_info_columns]
    .drop_duplicates(subset="CourseID")
)

print("Course information shape:")
print(course_info.shape)

Course information shape:
(60, 13)


## Creating the final target datasets

In [18]:
prediction_targets = course_info.merge(
    targets,
    on="CourseID",
    how="inner"
)

print("Final prediction target dataset:")
print(prediction_targets.shape)

prediction_targets.head()

Final prediction target dataset:
(60, 15)


,CourseID,CourseName,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating,TeacherID,TeacherName,TeacherRating,YearsOfExperience,Expertise,EnrollmentCount,CourseRevenue
0,CR00050,Computer Vision,Artificial Intelligence,Paid,Beginner,490.9,7.55,4.55,TC00040,Kimberly Miller,4.58,24,Cybersecurity,174,85416.6
1,CR00021,Data Analysis with Python,Data Science,Free,Intermediate,0.0,1.20,3.60,TC00016,David Carlson,2.92,1,Data Science,196,0.0
2,CR00009,Web Design Fundamentals,Design,Free,Beginner,0.0,48.19,4.51,TC00051,John Obrien,1.77,2,Design,155,0.0
3,CR00022,Data Visualization,Data Science,Free,Beginner,0.0,32.64,3.65,TC00010,Frances Sanchez,2.18,1,Data Science,177,0.0
4,CR00027,Neural Networks,Machine Learning,Free,Advanced,0.0,9.30,1.81,TC00036,Brenda Mclean,1.39,4,Digital Marketing,152,0.0


## Checking target distributions

In [19]:
print("===== ENROLLMENT COUNT =====")
print(prediction_targets["EnrollmentCount"].describe())

print("\n===== COURSE REVENUE =====")
print(prediction_targets["CourseRevenue"].describe())

===== ENROLLMENT COUNT =====
count     60.000000
mean     166.666667
std       12.523424
min      140.000000
25%      157.500000
50%      166.000000
75%      174.750000
max      196.000000
Name: EnrollmentCount, dtype: float64

===== COURSE REVENUE =====
count       60.000000
mean     15188.724500
std      25409.721911
min          0.000000
25%          0.000000
50%          0.000000
75%      20858.060000
max      85416.600000
Name: CourseRevenue, dtype: float64


In [20]:
print(
    "EnrollmentCount skewness:",
    prediction_targets["EnrollmentCount"].skew()
)

print(
    "CourseRevenue skewness:",
    prediction_targets["CourseRevenue"].skew()
)

EnrollmentCount skewness: 0.15652075921589667
CourseRevenue skewness: 1.5037561345104173


## Checking zero values

In [21]:
print(
    "Courses with zero EnrollmentCount:",
    (prediction_targets["EnrollmentCount"] == 0).sum()
)

print(
    "Courses with zero CourseRevenue:",
    (prediction_targets["CourseRevenue"] == 0).sum()
)

Courses with zero EnrollmentCount: 0
Courses with zero CourseRevenue: 38


In [22]:
prediction_targets[
    (prediction_targets["EnrollmentCount"] == 0) |
    (prediction_targets["CourseRevenue"] == 0)
]

,CourseID,CourseName,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating,TeacherID,TeacherName,TeacherRating,YearsOfExperience,Expertise,EnrollmentCount,CourseRevenue
1,CR00021,Data Analysis with Python,Data Science,Free,Intermediate,0.0,1.20,3.60,TC00016,David Carlson,2.92,1,Data Science,196,0.0
2,CR00009,Web Design Fundamentals,Design,Free,Beginner,0.0,48.19,4.51,TC00051,John Obrien,1.77,2,Design,155,0.0
3,CR00022,Data Visualization,Data Science,Free,Beginner,0.0,32.64,3.65,TC00010,Frances Sanchez,2.18,1,Data Science,177,0.0
4,CR00027,Neural Networks,Machine Learning,Free,Advanced,0.0,9.30,1.81,TC00036,Brenda Mclean,1.39,4,Digital Marketing,152,0.0
5,CR00036,Agile Project Management,Project Management,Free,Beginner,0.0,15.75,4.68,TC00040,Kimberly Miller,4.58,24,Cybersecurity,177,0.0
6,CR00018,Social Media Marketing,Marketing,Free,Advanced,0.0,15.44,2.01,TC00053,Aaron Kirby,4.29,9,Marketing,163,0.0
7,CR00037,Scrum Essentials,Project Management,Free,Intermediate,0.0,33.93,3.45,TC00040,Kimberly Miller,4.58,24,Cybersecurity,153,0.0
8,CR00060,Content Creation,Digital Marketing,Free,Beginner,0.0,8.95,2.14,TC00036,Brenda Mclean,1.39,4,Digital Marketing,165,0.0
9,CR00053,React for Beginners,Web Development,Free,Advanced,0.0,40.07,2.67,TC00042,Yolanda Levine,4.97,21,Machine Learning,163,0.0
10,CR00003,C++ for Beginners,Programming,Free,Beginner,0.0,19.53,3.85,TC00052,Susan Johnson,3.43,12,Business,173,0.0


## Target correlation

In [23]:
target_correlation = prediction_targets[
    ["EnrollmentCount", "CourseRevenue"]
].corr()

target_correlation

,EnrollmentCount,CourseRevenue
EnrollmentCount,1.000000,-0.116611
CourseRevenue,-0.116611,1.000000


## Creating Target Summary

In [24]:
target_summary = pd.DataFrame({
    "Target": [
        "EnrollmentCount",
        "CourseRevenue"
    ],
    "Description": [
        "Number of transactions/enrollments associated with each course",
        "Total revenue generated by each course"
    ],
    "DataType": [
        prediction_targets["EnrollmentCount"].dtype,
        prediction_targets["CourseRevenue"].dtype
    ],
    "MissingValues": [
        prediction_targets["EnrollmentCount"].isnull().sum(),
        prediction_targets["CourseRevenue"].isnull().sum()
    ],
    "UniqueValues": [
        prediction_targets["EnrollmentCount"].nunique(),
        prediction_targets["CourseRevenue"].nunique()
    ]
})

target_summary

,Target,Description,DataType,MissingValues,UniqueValues
0,EnrollmentCount,Number of transactions/enrollments associated ...,int64,0,36
1,CourseRevenue,Total revenue generated by each course,float64,0,23


## Final Validation

In [25]:
print("========== DAY 8 TARGET VALIDATION ==========")

print("\nDataset shape:")
print(prediction_targets.shape)

print("\nNumber of unique courses:")
print(prediction_targets["CourseID"].nunique())

print("\nMissing values:")
print(prediction_targets.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(prediction_targets["CourseID"].duplicated().sum())

print("\nTotal EnrollmentCount:")
print(prediction_targets["EnrollmentCount"].sum())

print("\nTotal CourseRevenue:")
print(prediction_targets["CourseRevenue"].sum())

print("\nOriginal transaction rows:")
print(len(features))

print("\nTarget columns:")
print(
    prediction_targets[
        ["EnrollmentCount", "CourseRevenue"]
    ].columns.tolist()
)

========== DAY 8 TARGET VALIDATION ==========

Dataset shape:
(60, 15)

Number of unique courses:
60

Missing values:
0

Duplicate CourseIDs:
0

Total EnrollmentCount:
10000

Total CourseRevenue:
911323.4700000001

Original transaction rows:
10000

Target columns:
['EnrollmentCount', 'CourseRevenue']


## Saving outputs (Day 8) 

In [26]:
target_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data"
)

target_folder.mkdir(
    parents=True,
    exist_ok=True
)

target_file_path = (
    target_folder
    / "EduPro_Day8_Prediction_Targets.xlsx"
)

In [27]:
with pd.ExcelWriter(
    target_file_path,
    engine="openpyxl"
) as writer:

    prediction_targets.to_excel(
        writer,
        sheet_name="Prediction_Targets",
        index=False
    )

    target_summary.to_excel(
        writer,
        sheet_name="Target_Summary",
        index=False
    )

    targets.to_excel(
        writer,
        sheet_name="Targets_Only",
        index=False
    )

print("✅ Day 8 Prediction Targets saved successfully:")
print(target_file_path)

✅ Day 8 Prediction Targets saved successfully:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day8_Prediction_Targets.xlsx


## Reloading & Verifing the Save File

In [28]:
check_targets = pd.read_excel(
    target_file_path,
    sheet_name="Prediction_Targets"
)

print("Saved target dataset:")
print(check_targets.shape)

print("\nMissing values:")
print(check_targets.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(check_targets["CourseID"].duplicated().sum())

print("\nTargets:")
print(
    check_targets[
        ["EnrollmentCount", "CourseRevenue"]
    ].head()
)

Saved target dataset:
(60, 15)

Missing values:
0

Duplicate CourseIDs:
0

Targets:
   EnrollmentCount  CourseRevenue
0              174        85416.6
1              196            0.0
2              155            0.0
3              177            0.0
4              152            0.0
